In [1]:
from langchain_community.document_loaders import PyPDFLoader
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
load_dotenv()

llm = ChatOpenAI(model = 'gpt-4o-mini')

/var/folders/fq/kr6gv5l17pd4j572n0_n3f2c0000gn/T/ipykernel_5738/2850576360.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/Users/rahultiwari/Documents/02_Freelancing/Hachion_batch/ai_engineering_19th_may/ai-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# step 1 : load the data
loader = PyPDFLoader("bajaj_finance_policy_prose_v1.pdf")
pages = loader.load()
# pages

In [ ]:
# step 2: Create chunks ( it will not work in productions ) --> RecursiveCharacterSpliter
from langchain_text_splitters import CharacterTextSplitter

splitter = CharacterTextSplitter(chunk_size=500, separator=".",
                                chunk_overlap=20)


chunks = splitter.split_documents(pages)

In [56]:
# step 3: Vectorisations ??
# embedding model ??
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7343.54it/s]


In [59]:
chunks[0].metadata

{'producer': 'ReportLab PDF Library - (opensource)',
 'creator': '(unspecified)',
 'creationdate': '2026-05-28T12:50:55+00:00',
 'author': 'BajajBot Training Data',
 'keywords': '',
 'moddate': '2026-05-28T12:50:55+00:00',
 'subject': '(unspecified)',
 'title': 'Bajaj Finance Helpdesk Knowledge Base — Prose Edition',
 'trapped': '/False',
 'source': 'bajaj_finance_policy_prose_v1.pdf',
 'total_pages': 14,
 'page': 0,
 'page_label': '1'}

In [65]:
# print(chunks[1].page_content)

chunk_text = [c.page_content for c in chunks]
chunk_metadata = [c.metadata for c in chunks]

In [66]:
chunk_vectors = model.encode(chunk_text,
                             show_progress_bar=True,
                            normalize_embeddings=True,
                            convert_to_numpy=True)


Batches: 100%|██████████| 3/3 [00:01<00:00,  2.69it/s]


In [69]:
len(chunk_vectors[0])

384

In [73]:
# store ??? 
store = {
    "vectors": chunk_vectors,
    "texts": chunk_text,
    "metadata": chunk_metadata,
    "model": "all-MiniLM-L6-v2"
        }

import pickle

with open("vector_store.pkl", "wb") as f:
    pickle.dump(store, f)

In [3]:
# load the vector store
import pickle
with open("vector_store.pkl", "rb") as f:
    store = pickle.load(f)

chunk_vectors = store["vectors"]
chunk_text = store["texts"]
chunk_metadata = store["metadata"]
model_name = store["model"]


In [6]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10219.36it/s]


In [ ]:
# step 5 : Build retrival functions ??
def search_similarity(query, top_k=3):
    # Convert the query into an embedding
    query_vector = model.encode([query], convert_to_numpy=True)

    # Compare query with all chunk vectors using cosine similarity
    scores = cosine_similarity(query_vector, chunk_vectors)[0]

    # Get indexes of top matching chunks
    top_indexes = scores.argsort()[::-1][:top_k]

   
    results = []
    for i in top_indexes:
        results.append((float(scores[i]), int(i)))

    return results

q_relevant = "What is the foreclosure charge on gold loans?"

search_results = search_similarity(q_relevant)
search_results

[(0.615541934967041, 71), (0.5716135501861572, 70), (0.5700260400772095, 69)]

In [10]:
print(chunk_text[71])

For closures between six and twelve
months, the charge further reduces to 1 percent of the outstanding principal. After 12 months, there is
no foreclosure charge — the customer pays only the outstanding principal plus interest accrued to date.
For all closures, a branch visit is required and the NOC is issued within 30 minutes.
7.3 Gold Auction Policy
If a customer fails to repay a gold loan or renew it at maturity, Bajaj Finance sends a formal written
notice


In [11]:
print(chunk_text[70])

If the loan is closed within the first three months, a foreclosure charge of 2 percent on the outstanding
principal plus all accrued interest applies. A branch visit is mandatory for this closure, and the customer
must present the original pledge receipt.
For loans closed between three and six months after disbursement, the charge reduces to 2 percent on
the outstanding principal only, with no accrued interest component


In [12]:
print(chunk_text[69])

Disbursal is completed on the same day,
typically within 30 minutes of appraisal and KYC completion. Pledged gold is stored in bank-grade
vaults that are fully insured and monitored by 24-hour CCTV surveillance.
7.2 Gold Loan Foreclosure Policy
Gold loan foreclosure refers to early closure of the loan before the completion of the agreed tenure. This
is the most frequently raised query from gold loan customers, so agents must be completely clear on the
applicable charges


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


GOOD_PROMPT = ChatPromptTemplate.from_template("""
You are BajajBot, an AI assistant for Bajaj Finance helpdesk agents.

CONTEXT (retrieved from Bajaj Finance policy documents):
-------------------------------------------------------
{context}
-------------------------------------------------------

INSTRUCTIONS:
- Answer ONLY using the CONTEXT above.
- Do NOT use your training knowledge.
- If the answer is not in the CONTEXT, say exactly:
  "I don't have this information in the provided documents."
- Keep your answer under 3 sentences.
- If a specific number or rule is in CONTEXT, include it.

QUESTION: {question}

ANSWER:
""")

In [15]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

chain = GOOD_PROMPT | llm | StrOutputParser()

In [16]:
def search_similarity(query, top_k=3):
    query_vector = model.encode([query], convert_to_numpy=True)
    scores = cosine_similarity(query_vector, chunk_vectors)[0]
    top_indexes = scores.argsort()[::-1][:top_k]

    results = []
    for i in top_indexes:
        results.append((float(scores[i]), int(i)))

    return results



def get_context(query, top_k=3):
    results = search_similarity(query, top_k=top_k)
    print(results)

    context_parts = []
    for score, idx in results:
        text = chunk_text[idx]
        context_parts.append(text)

    print(context_parts)

    context = "\n\n".join(context_parts)
    return context

In [24]:
q_relevant = "who is pm of India?"

context = get_context(q_relevant)
# print(search_results)

[(0.18795305490493774, 73), (0.170728862285614, 53), (0.16986338794231415, 2)]
['1 How Bajaj Finance Reports to Credit Bureaus\nBajaj Finance reports loan data to all four major credit bureaus operating in India: TransUnion CIBIL,\nExperian, Equifax, and CRIF High Mark. Reporting is done monthly, on the 7th of every month, covering', 'Section 6 — Business Loan: SME and MSME Products\n6.1 Business Loan Product Overview\nBajaj Finance offers business loans to SMEs and MSMEs in two broad categories: unsecured loans up\nto Rs 50 lakhs and secured loans up to Rs 2 crores. Unsecured loans do not require any collateral,\nwhile secured loans require acceptable security as defined in Section 6.4.\nFor unsecured loans, the tenure ranges from 12 to 96 months. Secured business loans can extend up to\n120 months', 'However, all eligibility conditions must be satisfied simultaneously — meeting only some\nof the criteria is not sufficient for approval.\nFor salaried applicants, the minimum age is 21 

In [25]:
answer = chain.invoke({"context": context, "question": q_relevant})

In [26]:
print(answer)

I don't have this information in the provided documents.


In [44]:
SAMPLE_TEXT="""GOLD LOAN POLICY — BAJAJ FINANCE

Gold purity accepted: 18 karat to 24 karat.
Maximum LTV: 75 percent of market value (RBI mandated).
Tenure: 3 months to 24 months.

Foreclosure Policy:
Within 3 months: 2 percent foreclosure charge
3 to 6 months: 2 percent charge
6 to 12 months: 1 percent charge
After 12 months: No foreclosure charge

Disbursal: Same day within 30 minutes of appraisal.
"""


In [45]:
output = splitter.split_text(SAMPLE_TEXT)

Created a chunk of size 223, which is longer than the specified 100


In [46]:
output

['GOLD LOAN POLICY — BAJAJ FINANCE\n\nGold purity accepted: 18 karat to 24 karat',
 'Maximum LTV: 75 percent of market value (RBI mandated).\nTenure: 3 months to 24 months',
 'Foreclosure Policy:\nWithin 3 months: 2 percent foreclosure charge\n3 to 6 months: 2 percent charge\n6 to 12 months: 1 percent charge\nAfter 12 months: No foreclosure charge\n\nDisbursal: Same day within 30 minutes of appraisal']

In [47]:
len(output[2])

221

## Experimentations

In [8]:
print(pages[0].page_content)

BAJAJ FINANCE LIMITED
 Helpdesk Agent Knowledge Base
 Prose Reference Edition — FY 2024–25
 Document Type: Internal Training & Reference Manual
 Coverage: Personal Loan · Home Loan · Gold Loan · Business Loan · CIBIL Policy
 Intended Users: Helpdesk Agents · Branch Executives · Collections Team
 Classification: CONFIDENTIAL — For Internal Use Only
 Version: v1.0 Prose Edition — May 2025
This document is the prose-format reference edition of the Bajaj Finance Helpdesk Knowledge Base. All
policy information is presented in descriptive paragraph form to support agent training, onboarding, and
the BajajBot AI assistant knowledge base. For structured lookup tables, refer to the companion Policy
Reference Document v4.0.
Section 1 — Personal Loan: Eligibility, Rates & Charges
1.1 Who Can Apply for a Bajaj Finance Personal Loan
Bajaj Finance personal loans are designed for both salaried employees and self-employed
professionals. However, all eligibility conditions must be satisfied simultaneou

In [18]:
chunk_size =100

output = []

for i in range(0, len(SAMPLE_TEXT), chunk_size):
    output.append(SAMPLE_TEXT[i:i + chunk_size])

output


['GOLD LOAN POLICY — BAJAJ FINANCE\n\nGold purity accepted: 18 karat to 24 karat.\nMaximum LTV: 75 percen',
 't of market value (RBI mandated).\nTenure: 3 months to 24 months.\n\nForeclosure Policy:\nWithin 3 month',
 's: 2 percent foreclosure charge\n3 to 6 months: 2 percent charge\n6 to 12 months: 1 percent charge\nAft',
 'er 12 months: No foreclosure charge\n\nDisbursal: Same day within 30 minutes of appraisal.\n']